# 04 — Transform Payments to Silver

## Purpose

Transform validated Bronze payment-attempt events into a clean Silver payment table while preserving retry history.

This notebook will standardize payment attributes, validate invoice ownership, enforce payment-attempt sequencing and settlement rules, and persist the result through an idempotent Delta merge.

## Sources

- `workspace.revenue_leakage_bronze.payment_events`
- `workspace.revenue_leakage_silver.invoices`

## Target

- `workspace.revenue_leakage_silver.payments`

## 1. Load and Inspect Payment Sources

Load the Bronze payment events and the Silver invoice reference table, then inspect the exact payment schema, status distribution, and retry-attempt volumes.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_PAYMENTS_TABLE = (
    "workspace.revenue_leakage_bronze.payment_events"
)

SILVER_INVOICES_TABLE = (
    "workspace.revenue_leakage_silver.invoices"
)

SILVER_PAYMENTS_TABLE = (
    "workspace.revenue_leakage_silver.payments"
)

bronze_payment_events_df = spark.table(
    BRONZE_PAYMENTS_TABLE
)

silver_invoice_reference_df = spark.table(
    SILVER_INVOICES_TABLE
)

print(
    f"Bronze payment events: "
    f"{bronze_payment_events_df.count():,}"
)

print(
    f"Silver invoice references: "
    f"{silver_invoice_reference_df.count():,}"
)

bronze_payment_events_df.printSchema()

display(
    bronze_payment_events_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

## 2. Standardize and Resolve Payment Attempts

Standardize payment attributes, remove exact duplicate events, and resolve the latest state of every payment attempt.

Every retained payment must reference an existing Silver invoice, and its subscription and customer identifiers must match the ownership recorded on that invoice.

In [0]:
payment_cdc_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col(
            "_source_file_modification_time"
        ).desc(),
        F.col("_ingested_at").desc(),
        F.col("_record_hash").desc(),
    )
)

standardized_payment_events_df = (
    bronze_payment_events_df
    .dropDuplicates(["_record_hash"])
    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id")),
    )
    .withColumn(
        "provider_transaction_id",
        F.trim(
            F.col("provider_transaction_id")
        ),
    )
    .withColumn(
        "invoice_id",
        F.trim(F.col("invoice_id")),
    )
    .withColumn(
        "subscription_id",
        F.trim(F.col("subscription_id")),
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")),
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency"))),
    )
    .withColumn(
        "payment_status",
        F.initcap(
            F.trim(F.col("payment_status"))
        ),
    )
    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method")),
    )
    .withColumn(
        "payment_provider",
        F.trim(F.col("payment_provider")),
    )
    .withColumn(
        "failure_reason",
        F.trim(F.col("failure_reason")),
    )
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation"))),
    )
)

latest_payment_events_df = (
    standardized_payment_events_df
    .withColumn(
        "_cdc_rank",
        F.row_number().over(
            payment_cdc_window
        ),
    )
    .filter(F.col("_cdc_rank") == 1)
    .drop("_cdc_rank")
)

active_payment_states_df = (
    latest_payment_events_df
    .filter(F.col("operation") != "DELETE")
)

invoice_ownership_df = (
    silver_invoice_reference_df
    .select(
        "invoice_id",
        F.col("subscription_id").alias(
            "_reference_subscription_id"
        ),
        F.col("customer_id").alias(
            "_reference_customer_id"
        ),
    )
    .dropDuplicates(["invoice_id"])
)

payment_reference_check_df = (
    active_payment_states_df
    .join(
        invoice_ownership_df,
        on="invoice_id",
        how="left",
    )
)

orphan_payment_states_df = (
    payment_reference_check_df
    .filter(
        F.col(
            "_reference_subscription_id"
        ).isNull()
    )
)

ownership_mismatch_payment_states_df = (
    payment_reference_check_df
    .filter(
        F.col(
            "_reference_subscription_id"
        ).isNotNull()
        & (
            (
                F.col("subscription_id")
                != F.col(
                    "_reference_subscription_id"
                )
            )
            | (
                F.col("customer_id")
                != F.col(
                    "_reference_customer_id"
                )
            )
        )
    )
)

silver_payments_df = (
    payment_reference_check_df
    .filter(
        F.col(
            "_reference_subscription_id"
        ).isNotNull()
        & (
            F.col("subscription_id")
            == F.col(
                "_reference_subscription_id"
            )
        )
        & (
            F.col("customer_id")
            == F.col(
                "_reference_customer_id"
            )
        )
    )
    .select(
        "payment_id",
        "provider_transaction_id",
        "invoice_id",
        "subscription_id",
        "customer_id",
        "transaction_amount",
        "settled_amount",
        "currency",
        "payment_status",
        "attempt_number",
        "payment_method",
        "payment_provider",
        "failure_reason",
        "payment_date",
        "settlement_date",
        F.col("operation").alias(
            "last_operation"
        ),
        F.col("event_timestamp").alias(
            "last_event_timestamp"
        ),
        "snapshot_date",
        "_source_system",
        "_source_entity",
        "_source_file_path",
        "_record_hash",
        F.current_timestamp().alias(
            "_silver_processed_at"
        ),
    )
)

print(
    f"Bronze payment events: "
    f"{bronze_payment_events_df.count():,}"
)

print(
    f"Distinct Bronze events: "
    f"{standardized_payment_events_df.count():,}"
)

print(
    f"Latest payment states: "
    f"{latest_payment_events_df.count():,}"
)

print(
    f"Payments without a Silver invoice: "
    f"{orphan_payment_states_df.count():,}"
)

print(
    f"Payment ownership mismatches: "
    f"{ownership_mismatch_payment_states_df.count():,}"
)

print(
    f"Eligible Silver payments: "
    f"{silver_payments_df.count():,}"
)

display(
    silver_payments_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

## 3. Profile Payment Business Domains

Inspect the standardized payment statuses, methods, providers, currencies, and failure reasons before enforcing explicit Silver data-quality rules.

In [0]:
payment_status_values = sorted(
    row["payment_status"]
    for row in (
        silver_payments_df
        .select("payment_status")
        .distinct()
        .collect()
    )
)

payment_method_values = sorted(
    row["payment_method"]
    for row in (
        silver_payments_df
        .select("payment_method")
        .distinct()
        .collect()
    )
)

payment_provider_values = sorted(
    row["payment_provider"]
    for row in (
        silver_payments_df
        .select("payment_provider")
        .distinct()
        .collect()
    )
)

payment_currency_values = sorted(
    row["currency"]
    for row in (
        silver_payments_df
        .select("currency")
        .distinct()
        .collect()
    )
)

failure_reason_values = sorted(
    row["failure_reason"]
    for row in (
        silver_payments_df
        .filter(
            F.col("failure_reason").isNotNull()
        )
        .select("failure_reason")
        .distinct()
        .collect()
    )
)

print(
    f"Payment statuses: "
    f"{payment_status_values}"
)

print(
    f"Payment methods: "
    f"{payment_method_values}"
)

print(
    f"Payment providers: "
    f"{payment_provider_values}"
)

print(
    f"Currencies: "
    f"{payment_currency_values}"
)

print(
    f"Failure reasons: "
    f"{failure_reason_values}"
)

display(
    silver_payments_df
    .groupBy(
        "payment_provider",
        "payment_status",
    )
    .count()
    .orderBy(
        "payment_provider",
        "payment_status",
    )
)

## 4. Validate the Silver Payment Attempts

Validate payment identifiers, invoice ownership, business domains, dates, amounts, settlement states, and retry sequencing.

Successful settlements must reconcile exactly with paid Silver invoices. Failed retries must follow an existing failed first attempt, while pending and failed payments must not contain settled amounts.

In [0]:
EXPECTED_SILVER_PAYMENT_COUNT = 29_972
EXPECTED_EXCLUDED_PAYMENT_COUNT = 260

ALLOWED_PAYMENT_STATUSES = [
    "Failed",
    "Pending",
    "Succeeded",
]

ALLOWED_PAYMENT_METHODS = [
    "ACH",
    "Bank Transfer",
    "Credit Card",
]

ALLOWED_PAYMENT_PROVIDERS = [
    "Adyen",
    "Stripe",
]

ALLOWED_PAYMENT_CURRENCIES = [
    "USD",
]

ALLOWED_FAILURE_REASONS = [
    "Bank Rejected",
    "Card Declined",
    "Insufficient Funds",
    "Processor Timeout",
]

required_payment_columns = [
    "payment_id",
    "provider_transaction_id",
    "invoice_id",
    "subscription_id",
    "customer_id",
    "transaction_amount",
    "settled_amount",
    "currency",
    "payment_status",
    "attempt_number",
    "payment_method",
    "payment_provider",
    "payment_date",
    "last_event_timestamp",
    "snapshot_date",
]

required_payment_field_is_missing = None

for column_name in required_payment_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == ""
        )
    )

    required_payment_field_is_missing = (
        missing_condition
        if required_payment_field_is_missing
        is None
        else required_payment_field_is_missing
        | missing_condition
    )

invalid_payment_status_condition = (
    ~F.col("payment_status").isin(
        ALLOWED_PAYMENT_STATUSES
    )
)

invalid_payment_method_condition = (
    ~F.col("payment_method").isin(
        ALLOWED_PAYMENT_METHODS
    )
)

invalid_payment_provider_condition = (
    ~F.col("payment_provider").isin(
        ALLOWED_PAYMENT_PROVIDERS
    )
)

invalid_payment_currency_condition = (
    ~F.col("currency").isin(
        ALLOWED_PAYMENT_CURRENCIES
    )
)

invalid_failure_reason_condition = (
    F.col("failure_reason").isNotNull()
    & ~F.col("failure_reason").isin(
        ALLOWED_FAILURE_REASONS
    )
)

invalid_payment_operation_condition = (
    ~F.col("last_operation").isin(
        "INSERT",
        "UPDATE",
    )
)

invalid_attempt_number_condition = (
    ~F.col("attempt_number").isin(1, 2)
)

invalid_payment_date_condition = (
    (
        F.col("payment_date")
        > F.col("snapshot_date")
    )
    | (
        F.col("settlement_date").isNotNull()
        & (
            (
                F.col("settlement_date")
                < F.col("payment_date")
            )
            | (
                F.col("settlement_date")
                > F.col("snapshot_date")
            )
        )
    )
)

invalid_payment_amount_condition = (
    (F.col("transaction_amount") <= 0)
    | (F.col("settled_amount") < 0)
    | (
        F.col("settled_amount")
        > F.col("transaction_amount")
    )
)

invalid_payment_state_condition = (
    (
        F.col("payment_status")
        == "Succeeded"
    )
    & (
        F.col("settlement_date").isNull()
        | (
            F.abs(
                F.col("settled_amount")
                - F.col("transaction_amount")
            )
            > F.lit(0.01)
        )
        | F.col("failure_reason").isNotNull()
    )
) | (
    (
        F.col("payment_status")
        == "Failed"
    )
    & (
        F.col("failure_reason").isNull()
        | (
            F.trim(
                F.col("failure_reason")
            )
            == ""
        )
        | F.col("settlement_date").isNotNull()
        | (F.col("settled_amount") != 0)
    )
) | (
    (
        F.col("payment_status")
        == "Pending"
    )
    & (
        F.col("failure_reason").isNotNull()
        | F.col("settlement_date").isNotNull()
        | (F.col("settled_amount") != 0)
    )
)

payment_validation_metrics = (
    silver_payments_df
    .agg(
        F.count("*").alias(
            "silver_payment_count"
        ),
        F.countDistinct("payment_id").alias(
            "distinct_payment_count"
        ),
        F.countDistinct(
            "provider_transaction_id"
        ).alias(
            "distinct_provider_transaction_count"
        ),
        F.sum(
            F.when(
                required_payment_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_field_count"),
        F.sum(
            F.when(
                invalid_payment_status_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_status_count"),
        F.sum(
            F.when(
                invalid_payment_method_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_method_count"),
        F.sum(
            F.when(
                invalid_payment_provider_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_provider_count"),
        F.sum(
            F.when(
                invalid_payment_currency_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_currency_count"),
        F.sum(
            F.when(
                invalid_failure_reason_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_failure_reason_count"),
        F.sum(
            F.when(
                invalid_payment_operation_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_operation_count"),
        F.sum(
            F.when(
                invalid_attempt_number_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_attempt_number_count"),
        F.sum(
            F.when(
                invalid_payment_date_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_date_count"),
        F.sum(
            F.when(
                invalid_payment_amount_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_amount_count"),
        F.sum(
            F.when(
                invalid_payment_state_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_payment_state_count"),
    )
    .first()
    .asDict()
)

duplicate_payment_count = (
    payment_validation_metrics[
        "silver_payment_count"
    ]
    - payment_validation_metrics[
        "distinct_payment_count"
    ]
)

duplicate_provider_transaction_count = (
    payment_validation_metrics[
        "silver_payment_count"
    ]
    - payment_validation_metrics[
        "distinct_provider_transaction_count"
    ]
)

duplicate_invoice_attempt_count = (
    silver_payments_df
    .groupBy(
        "invoice_id",
        "attempt_number",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

payment_retry_window = (
    Window.partitionBy("invoice_id")
)

payment_retry_validation_df = (
    silver_payments_df
    .withColumn(
        "_attempt_one_count",
        F.sum(
            F.when(
                F.col("attempt_number") == 1,
                1,
            ).otherwise(0)
        ).over(payment_retry_window),
    )
    .withColumn(
        "_attempt_one_status",
        F.max(
            F.when(
                F.col("attempt_number") == 1,
                F.col("payment_status"),
            )
        ).over(payment_retry_window),
    )
)

invalid_retry_sequence_count = (
    payment_retry_validation_df
    .filter(
        (F.col("attempt_number") == 2)
        & (
            (F.col("_attempt_one_count") != 1)
            | (
                F.col("_attempt_one_status")
                != "Failed"
            )
        )
    )
    .count()
)

invoice_payment_reference_df = (
    silver_invoice_reference_df
    .select(
        "invoice_id",
        F.col("invoice_status").alias(
            "_invoice_status"
        ),
        F.col("invoice_total_amount").alias(
            "_invoice_total_amount"
        ),
    )
)

payments_with_invoice_df = (
    silver_payments_df
    .join(
        invoice_payment_reference_df,
        on="invoice_id",
        how="inner",
    )
)

invalid_invoice_amount_count = (
    payments_with_invoice_df
    .filter(
        F.abs(
            F.col("transaction_amount")
            - F.col("_invoice_total_amount")
        )
        > F.lit(0.01)
    )
    .count()
)

invalid_invoice_status_mapping_count = (
    payments_with_invoice_df
    .filter(
        (
            (
                F.col("payment_status")
                == "Succeeded"
            )
            & (
                F.col("_invoice_status")
                != "Paid"
            )
        )
        | (
            (
                F.col("payment_status")
                == "Pending"
            )
            & (
                F.col("_invoice_status")
                != "Open"
            )
        )
        | (
            (
                F.col("payment_status")
                == "Failed"
            )
            & ~F.col("_invoice_status").isin(
                "Paid",
                "Past Due",
            )
        )
    )
    .count()
)

successful_payment_invoice_df = (
    silver_payments_df
    .filter(
        F.col("payment_status")
        == "Succeeded"
    )
    .select("invoice_id")
)

paid_invoice_without_success_count = (
    silver_invoice_reference_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .select("invoice_id")
    .join(
        successful_payment_invoice_df,
        on="invoice_id",
        how="left_anti",
    )
    .count()
)

duplicate_successful_invoice_count = (
    successful_payment_invoice_df
    .groupBy("invoice_id")
    .count()
    .filter(F.col("count") != 1)
    .count()
)

failed_payment_invoice_df = (
    silver_payments_df
    .filter(
        F.col("payment_status") == "Failed"
    )
    .select("invoice_id")
    .distinct()
)

past_due_invoice_without_failure_count = (
    silver_invoice_reference_df
    .filter(
        F.col("invoice_status") == "Past Due"
    )
    .select("invoice_id")
    .join(
        failed_payment_invoice_df,
        on="invoice_id",
        how="left_anti",
    )
    .count()
)

successful_settlement_mismatch_count = (
    payments_with_invoice_df
    .filter(
        F.col("payment_status")
        == "Succeeded"
    )
    .filter(
        F.abs(
            F.col("settled_amount")
            - F.col("_invoice_total_amount")
        )
        > F.lit(0.01)
    )
    .count()
)

excluded_payment_count = (
    orphan_payment_states_df.count()
)

ownership_mismatch_count = (
    ownership_mismatch_payment_states_df.count()
)

for metric_name, metric_value in (
    payment_validation_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

additional_payment_metrics = {
    "duplicate_payment_count":
        duplicate_payment_count,
    "duplicate_provider_transaction_count":
        duplicate_provider_transaction_count,
    "duplicate_invoice_attempt_count":
        duplicate_invoice_attempt_count,
    "invalid_retry_sequence_count":
        invalid_retry_sequence_count,
    "invalid_invoice_amount_count":
        invalid_invoice_amount_count,
    "invalid_invoice_status_mapping_count":
        invalid_invoice_status_mapping_count,
    "paid_invoice_without_success_count":
        paid_invoice_without_success_count,
    "duplicate_successful_invoice_count":
        duplicate_successful_invoice_count,
    "past_due_invoice_without_failure_count":
        past_due_invoice_without_failure_count,
    "successful_settlement_mismatch_count":
        successful_settlement_mismatch_count,
    "excluded_orphan_payment_count":
        excluded_payment_count,
    "ownership_mismatch_count":
        ownership_mismatch_count,
}

for metric_name, metric_value in (
    additional_payment_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

assert (
    payment_validation_metrics[
        "silver_payment_count"
    ]
    == EXPECTED_SILVER_PAYMENT_COUNT
), "Unexpected Silver payment count."

assert (
    excluded_payment_count
    == EXPECTED_EXCLUDED_PAYMENT_COUNT
), "Unexpected excluded payment count."

for metric_name in [
    "null_required_field_count",
    "invalid_status_count",
    "invalid_method_count",
    "invalid_provider_count",
    "invalid_currency_count",
    "invalid_failure_reason_count",
    "invalid_operation_count",
    "invalid_attempt_number_count",
    "invalid_date_count",
    "invalid_amount_count",
    "invalid_payment_state_count",
]:
    assert (
        payment_validation_metrics[
            metric_name
        ]
        == 0
    ), f"Validation failed: {metric_name}"

for metric_name, metric_value in (
    additional_payment_metrics.items()
):
    if metric_name != (
        "excluded_orphan_payment_count"
    ):
        assert metric_value == 0, (
            f"Validation failed: {metric_name}"
        )

print(
    "Silver payment validation completed successfully."
)

display(
    silver_payments_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .agg(
        F.count("*").alias("payment_count"),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias("transaction_amount"),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias("settled_amount"),
    )
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

## 5. Persist the Silver Payment Table

Persist the validated payment-attempt history as a managed Delta table.

The merge uses `payment_id` as the business key and updates a payment only when its source record hash changes. Every retry remains a separate record through its unique payment identifier and attempt number.

In [0]:
spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS
    workspace.revenue_leakage_silver
    """
)

silver_payments_df.createOrReplaceTempView(
    "silver_payment_updates"
)

if spark.catalog.tableExists(
    SILVER_PAYMENTS_TABLE
):
    spark.sql(
        f"""
        MERGE INTO
          {SILVER_PAYMENTS_TABLE} AS target
        USING
          silver_payment_updates AS source
        ON
          target.payment_id
          = source.payment_id

        WHEN MATCHED
          AND target._record_hash
              <> source._record_hash
        THEN
          UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *

        WHEN NOT MATCHED BY SOURCE THEN
          DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        silver_payments_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            SILVER_PAYMENTS_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )

saved_silver_payments_df = spark.table(
    SILVER_PAYMENTS_TABLE
)

saved_payment_count = (
    saved_silver_payments_df.count()
)

saved_distinct_payment_count = (
    saved_silver_payments_df
    .select("payment_id")
    .distinct()
    .count()
)

saved_distinct_transaction_count = (
    saved_silver_payments_df
    .select("provider_transaction_id")
    .distinct()
    .count()
)

saved_payment_reference_errors = (
    saved_silver_payments_df
    .join(
        invoice_ownership_df,
        on="invoice_id",
        how="left",
    )
    .filter(
        F.col(
            "_reference_subscription_id"
        ).isNull()
        | (
            F.col("subscription_id")
            != F.col(
                "_reference_subscription_id"
            )
        )
        | (
            F.col("customer_id")
            != F.col(
                "_reference_customer_id"
            )
        )
    )
    .count()
)

assert (
    saved_payment_count
    == EXPECTED_SILVER_PAYMENT_COUNT
), "Saved Silver payment count is incorrect."

assert (
    saved_distinct_payment_count
    == saved_payment_count
), "Saved table contains duplicate payments."

assert (
    saved_distinct_transaction_count
    == saved_payment_count
), "Saved provider transactions are duplicated."

assert (
    saved_payment_reference_errors
    == 0
), "Saved table contains reference errors."

print(
    f"Write method: {write_method}"
)

print(
    f"Silver table: "
    f"{SILVER_PAYMENTS_TABLE}"
)

print(
    f"Saved Silver payments: "
    f"{saved_payment_count:,}"
)

print(
    f"Saved distinct payment IDs: "
    f"{saved_distinct_payment_count:,}"
)

print(
    f"Saved distinct provider transactions: "
    f"{saved_distinct_transaction_count:,}"
)

print(
    f"Saved reference errors: "
    f"{saved_payment_reference_errors:,}"
)

display(
    saved_silver_payments_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .agg(
        F.count("*").alias("payment_count"),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias("transaction_amount"),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias("settled_amount"),
    )
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

## 6. Validate Idempotent Reprocessing

Reapply the same payment-attempt dataset through the Delta merge.

A successful rerun must produce zero inserts, updates, and deletes while preserving payment identifiers, retry history, and settlement totals.

In [0]:
rows_before_payment_rerun = (
    spark.table(
        SILVER_PAYMENTS_TABLE
    )
    .count()
)

spark.sql(
    f"""
    MERGE INTO
      {SILVER_PAYMENTS_TABLE} AS target
    USING
      silver_payment_updates AS source
    ON
      target.payment_id
      = source.payment_id

    WHEN MATCHED
      AND target._record_hash
          <> source._record_hash
    THEN
      UPDATE SET *

    WHEN NOT MATCHED THEN
      INSERT *

    WHEN NOT MATCHED BY SOURCE THEN
      DELETE
    """
)

latest_payment_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {SILVER_PAYMENTS_TABLE}
        LIMIT 1
        """
    )
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

latest_payment_history_row = (
    latest_payment_history_df.first()
)

payment_operation_metrics = (
    latest_payment_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted = int(
    payment_operation_metrics.get(
        "numTargetRowsInserted",
        "0",
    )
)

rows_updated = int(
    payment_operation_metrics.get(
        "numTargetRowsUpdated",
        "0",
    )
)

rows_deleted = int(
    payment_operation_metrics.get(
        "numTargetRowsDeleted",
        "0",
    )
)

payments_after_rerun_df = spark.table(
    SILVER_PAYMENTS_TABLE
)

rows_after_payment_rerun = (
    payments_after_rerun_df.count()
)

distinct_payments_after_rerun = (
    payments_after_rerun_df
    .select("payment_id")
    .distinct()
    .count()
)

distinct_transactions_after_rerun = (
    payments_after_rerun_df
    .select("provider_transaction_id")
    .distinct()
    .count()
)

duplicate_payments_after_rerun = (
    rows_after_payment_rerun
    - distinct_payments_after_rerun
)

duplicate_transactions_after_rerun = (
    rows_after_payment_rerun
    - distinct_transactions_after_rerun
)

payment_totals_after_rerun = (
    payments_after_rerun_df
    .agg(
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias("transaction_total"),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias("settlement_total"),
    )
    .first()
    .asDict()
)

assert (
    latest_payment_history_row["operation"]
    == "MERGE"
), "Latest Delta operation was not MERGE."

assert rows_inserted == 0, (
    "Idempotency failed: rows were inserted."
)

assert rows_updated == 0, (
    "Idempotency failed: rows were updated."
)

assert rows_deleted == 0, (
    "Idempotency failed: rows were deleted."
)

assert (
    rows_before_payment_rerun
    == EXPECTED_SILVER_PAYMENT_COUNT
), "Unexpected count before rerun."

assert (
    rows_after_payment_rerun
    == EXPECTED_SILVER_PAYMENT_COUNT
), "Unexpected count after rerun."

assert duplicate_payments_after_rerun == 0, (
    "Duplicate payments detected."
)

assert duplicate_transactions_after_rerun == 0, (
    "Duplicate provider transactions detected."
)

print(
    f"Rows before rerun: "
    f"{rows_before_payment_rerun:,}"
)

print(
    f"Rows after rerun: "
    f"{rows_after_payment_rerun:,}"
)

print(
    f"Rows inserted during rerun: "
    f"{rows_inserted:,}"
)

print(
    f"Rows updated during rerun: "
    f"{rows_updated:,}"
)

print(
    f"Rows deleted during rerun: "
    f"{rows_deleted:,}"
)

print(
    f"Duplicate payments: "
    f"{duplicate_payments_after_rerun:,}"
)

print(
    f"Duplicate provider transactions: "
    f"{duplicate_transactions_after_rerun:,}"
)

for metric_name, metric_value in (
    payment_totals_after_rerun.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,.2f}"
    )

print(
    "Silver payment transformation is idempotent."
)

display(
    latest_payment_history_df
)

## Result

The payment Silver transformation completed successfully:

- 30,232 Bronze payment attempts processed
- 260 payments without a current Silver invoice excluded
- 29,972 valid payment attempts persisted
- Zero duplicate payment identifiers
- Zero duplicate provider transaction identifiers
- Zero invoice ownership mismatches
- Zero invalid payment states or retry sequences
- Zero invoice amount or settlement mismatches
- Transaction-attempt total: 5,234,901.68 USD
- Successful settlement total: 3,806,266.54 USD
- Delta merge rerun produced zero changes
- Target confirmed as a managed Delta table